# libraries importing

In [10]:
import pandas as pd
import numpy as np
import re

# loading dataset to understand the structure

In [11]:
df=pd.read_csv("all_tickets_processed_improved_v3.csv")
df

,Document,Topic_group
0,connection with icon icon dear please setup ic...,Hardware
1,work experience user work experience user hi w...,Access
2,requesting for meeting requesting meeting hi p...,Hardware
3,reset passwords for external accounts re expir...,Access
4,mail verification warning hi has got attached ...,Miscellaneous
...,...,...
47832,git space for a project issues with adding use...,Access
47833,error sent july error hi guys can you help out...,Miscellaneous
47834,connection issues sent tuesday july connection...,Hardware
47835,error cube reports sent tuesday july error hel...,HR Support


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer #for text conversion
from sklearn.model_selection import train_test_split        #for training and testing dataset
from sklearn.linear_model import LogisticRegression         #Logistic regression
from sklearn.metrics import accuracy_score                  #accuracy
from sklearn.metrics import classification_report

In [13]:
df=df.rename(columns={
    "Document": "customer_ticket",
    "Topic_group":"category"
})

In [14]:
df.isnull().sum()

customer_ticket    0
category           0
dtype: int64

In [15]:
tfidf_vectorizer= TfidfVectorizer(
    stop_words='english',
    max_features=5000
)

In [16]:
X_features= tfidf_vectorizer.fit_transform(df['customer_ticket'])

In [17]:
y_labels=df['category']

In [18]:
X_train, X_test, y_train, y_test=train_test_split(
    X_features,
    y_labels,
    test_size=0.2,
    random_state=42
)

In [19]:
ticket_classifier=LogisticRegression(max_iter=1000)
ticket_classifier.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [20]:
y_predictions=ticket_classifier.predict(X_test)

In [21]:
print("Accuracy:", accuracy_score(y_test, y_predictions))
print("\nReport of Classification:\n", classification_report(y_test, y_predictions))

Accuracy: 0.845108695652174

Report of Classification:
                        precision    recall  f1-score   support

               Access       0.91      0.87      0.89      1455
Administrative rights       0.87      0.68      0.77       342
           HR Support       0.84      0.83      0.84      2107
             Hardware       0.79      0.88      0.83      2760
     Internal Project       0.91      0.80      0.85       451
        Miscellaneous       0.80      0.82      0.81      1400
             Purchase       0.97      0.88      0.92       497
              Storage       0.93      0.84      0.88       556

             accuracy                           0.85      9568
            macro avg       0.88      0.82      0.85      9568
         weighted avg       0.85      0.85      0.85      9568



In [31]:
def ticket_priority(customer_ticket):
    customer_ticket= customer_ticket.lower()
    if any(keyword in customer_ticket for keyword in['error','fail', 'crash', 'not','emergency', 'worried']):
        return 'High Priority'
    elif any(keyword in customer_ticket for keyword in['delay', 'request', 'change', 'instead', 'shift', 'rename']):
        return 'Medium Priority'
    else:
        return 'Low Priority'

In [32]:
def classify_ticket(ticket):
    ticket_vector=tfidf_vectorizer.transform([ticket])

    predicted_category=ticket_classifier.predict(ticket_vector)[0]

    predicted_priority= ticket_priority(ticket)
    return predicted_category, predicted_priority

In [36]:
classify_ticket("adding new printer xerox on th floor printer floor hi tried add printer floor show printers add manually help required steps thanks director central
")

('Hardware', 'High Priority')